In [1]:
import pandas as pd
import numpy as np
simulaciones = pd.read_json("simulation.jsonlines", lines=True)
planes = pd.read_json("plans.jsonlines", lines=True)

# Paso 2: Transformación de los datos

1. Aplico la función proporcionada en la página de Github

De este modo puedo ver con más claridad como están ordenados los datos, elimino la columna truck porque ya tenemos los datos en diferentes columnas.

In [2]:
planes = planes.join(planes.trucks.explode().apply(pd.Series), lsuffix='_sim').reset_index(drop=True).drop("trucks", axis=1)
planes

,items_sim,simulationId,items,route,truck_id
0,"[{'itemId': '0', 'locationId': '326ae8dc7810de...",a2bfd344-3b14-462a-82e7-d42aca54a650,"[23, 24, 4, 25, 26, 27, 28, 40, 41, 42, 43, 13...","[{'destination': 'af3ea5c0e98151a51bd39c64', '...",5534TPH
1,"[{'itemId': '0', 'locationId': '326ae8dc7810de...",a2bfd344-3b14-462a-82e7-d42aca54a650,"[74, 75, 76, 67, 68, 69, 70, 92, 93, 94, 55, 5...","[{'destination': 'faf06210580833f8c7963506', '...",2423VRT
2,"[{'itemId': '0', 'locationId': '326ae8dc7810de...",a2bfd344-3b14-462a-82e7-d42aca54a650,"[18, 19, 57, 58, 88, 89, 90, 91, 54, 20, 21, 2...","[{'destination': 'bef2df0471ee1c26b4ab6ba9', '...",0265TYL
3,"[{'itemId': '0', 'locationId': '326ae8dc7810de...",a2bfd344-3b14-462a-82e7-d42aca54a650,"[87, 0, 1, 2, 3, 33, 95, 96, 97]","[{'destination': '7a249403ad22a72495769f75', '...",8809GPH
4,"[{'itemId': '0', 'locationId': '326ae8dc7810de...",a2bfd344-3b14-462a-82e7-d42aca54a650,"[31, 32, 37, 38, 39, 66]","[{'destination': '1e38734ff239ccf75bf04b95', '...",1161GJN
...,...,...,...,...,...
635,"[{'itemId': '0', 'locationId': 'd1214b27823e6d...",6deec7b7-6c11-418c-a1bd-f99a9e3261b6,"[53, 54, 55, 83, 84, 85, 86, 26, 27, 29, 30, 3...","[{'destination': '225cd814768e8ca390983313', '...",5534TPH
636,"[{'itemId': '0', 'locationId': 'd1214b27823e6d...",6deec7b7-6c11-418c-a1bd-f99a9e3261b6,"[72, 73, 74, 88, 89, 90, 91, 43, 62, 63, 64, 3...","[{'destination': '417bf3aae83127cd905f3bc9', '...",3953RLD
637,"[{'itemId': '0', 'locationId': 'd1214b27823e6d...",6deec7b7-6c11-418c-a1bd-f99a9e3261b6,"[6, 7, 8, 9]","[{'destination': '1e1251e186dd507f63e4d1e6', '...",5030LXK
638,"[{'itemId': '0', 'locationId': 'd1214b27823e6d...",6deec7b7-6c11-418c-a1bd-f99a9e3261b6,"[44, 45, 46, 47, 94, 20, 21, 22, 95, 96, 97, 6...","[{'destination': '6a33d0bcd3624ccb86febf09', '...",3321FBL


In [3]:
planes = planes.groupby(["simulationId","truck_id"]).agg(list)

2. Ahora busco sacar información del dataframe de simulaciones

Observo que hay 8 tipos de eventos diferentes, "Truck departed" indica desde donde sale el camión al comienzo, "Truck initialized" indica cuando inicia el trabajo, "Truck received packets" indica la lista de paquetes indicando los paquetes y el lugar al que hay que enviarlos, "Truck arrived" indica a qué ubicación ha llegado, "Truck started delivering" indica el lugar al que está repartiendo y el paquete en concreto, "Truck ended delivering" indica cuándo ha terminado de repartir un paquete en concreto, "Truck departed to depot" indica el final de la jornada y "Truck ended route" indica que ha llegado al final de la ruta.

In [4]:
simulaciones = simulaciones.rename(columns={"truckId":"truck_id"})

In [6]:
lista_eventos = ["Truck departed","Truck arrived","Truck started delivering","Truck ended delivering"]

In [9]:
simulaciones[simulaciones.eventType.isin(lista_eventos)]

,eventDescription,eventTime,eventType,simulationId,truck_id
0,"(plaza del Dos de Mayo, 6, Madrid, [-3.7041862...",5760,Truck departed,a2bfd344-3b14-462a-82e7-d42aca54a650,1669HSZ
1,"(plaza del Dos de Mayo, 6, Madrid, [-3.7041862...",5730,Truck departed,a2bfd344-3b14-462a-82e7-d42aca54a650,8809GPH
2,"(plaza del Dos de Mayo, 6, Madrid, [-3.7041862...",5730,Truck departed,a2bfd344-3b14-462a-82e7-d42aca54a650,1161GJN
3,"(plaza del Dos de Mayo, 6, Madrid, [-3.7041862...",5430,Truck departed,a2bfd344-3b14-462a-82e7-d42aca54a650,3321FBL
4,"(plaza del Dos de Mayo, 6, Madrid, [-3.7041862...",5310,Truck departed,a2bfd344-3b14-462a-82e7-d42aca54a650,2423VRT
...,...,...,...,...,...
26572,"Packet 87 to (calle del General Pardiñas, 20, ...",11200680,Truck ended delivering,f0f90050-5a0f-4965-864e-d6bc0d02b5a0,8386WZB
26573,"(calle del General Pardiñas, 20, Madrid, [-3.6...",11200860,Truck departed,f0f90050-5a0f-4965-864e-d6bc0d02b5a0,8386WZB
26576,"Packet 83 to (calle de Juan de Austria, 1, Mad...",11562780,Truck started delivering,f0f90050-5a0f-4965-864e-d6bc0d02b5a0,8386WZB
26577,"(calle de Juan de Austria, 1, Madrid, [-3.6991...",11562780,Truck arrived,f0f90050-5a0f-4965-864e-d6bc0d02b5a0,8386WZB


In [11]:
simulaciones = simulaciones.groupby(["simulationId","truck_id"]).agg(list)

3. Junto los dos dataframes

In [12]:
conjunto = planes.join(simulaciones, lsuffix="_plan", rsuffix="_simul")

In [13]:
conjunto

items_sim  \
simulationId                         truck_id                                                      
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL   [[{'itemId': '0', 'locationId': 'bc451e2cea161...   
                                     3953RLD   [[{'itemId': '0', 'locationId': 'bc451e2cea161...   
                                     5030LXK   [[{'itemId': '0', 'locationId': 'bc451e2cea161...   
                                     5534TPH   [[{'itemId': '0', 'locationId': 'bc451e2cea161...   
                                     6270NFM   [[{'itemId': '0', 'locationId': 'bc451e2cea161...   
...                                                                                          ...   
fea13535-ade6-4215-96ca-dab5b4ef309b 0013DYS   [[{'itemId': '0', 'locationId': '17d843cfd0ff5...   
                                     3953RLD   [[{'itemId': '0', 'locationId': '17d843cfd0ff5...   
                                     5534TPH   [[{'itemId': '0', 'locationId': '17d843cfd0ff5...   
                                     6270NFM   [[{'itemId': '0', 'locationId': '17d843cfd0ff5...   
                                     6965XLY   [[{'itemId': '0', 'locationId': '17d843cfd0ff5...   

                                                                                           items  \
simulationId                         truck_id                                                      
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL   [[78, 79, 50, 51, 52, 23, 21, 58, 59, 60, 61, ...   
                                     3953RLD                                          [[44, 45]]   
                                     5030LXK          [[16, 17, 18, 19, 80, 81, 48, 38, 39, 22]]   
                                     5534TPH   [[68, 69, 70, 53, 54, 55, 40, 41, 42, 43, 73, ...   
                                     6270NFM         [[6, 7, 8, 35, 36, 37, 62, 63, 64, 65, 57]]   
...                                                                                          ...   
fea13535-ade6-4215-96ca-dab5b4ef309b 0013DYS                                            [[4, 5]]   
                                     3953RLD   [[71, 72, 73, 74, 53, 38, 39, 18, 70, 66, 67, ...   
                                     5534TPH   [[61, 62, 63, 11, 12, 13, 14, 60, 19, 20, 21, ...   
                                     6270NFM   [[36, 37, 9, 10, 17, 16, 56, 25, 26, 27, 28, 2...   
                                     6965XLY   [[15, 44, 45, 46, 47, 22, 23, 24, 2, 3, 40, 41...   

                                                                                           route  \
simulationId                         truck_id                                                      
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL   [[{'destination': '2e8d1b7b9fad4f083892d83c', ...   
                                     3953RLD   [[{'destination': '4bc556d929cc052029e14a78', ...   
                                     5030LXK   [[{'destination': '0550fd3ee7831440c5cf21c5', ...   
                                     5534TPH   [[{'destination': '3a471470f7f1d508af6cdfdb', ...   
                                     6270NFM   [[{'destination': '04463c1410164bb7855a830f', ...   
...                                                                                          ...   
fea13535-ade6-4215-96ca-dab5b4ef309b 0013DYS   [[{'destination': 'a840e3e2d217fe732e00f08c', ...   
                                     3953RLD   [[{'destination': '9492562782bb6f6befb7dfc8', ...   
                                     5534TPH   [[{'destination': '5a555ff3e81103468a36f757', ...   
                                     6270NFM   [[{'destination': '4a600d1ab9d6a40ea60dd088', ...   
                                     6965XLY   [[{'destination': '58bb022022a0e97ef0efcf46', ...   

                                                                                eventDescription  \
simulationId                         truck_id                                       

4. Vuelvo a aplicar la misma función pero esta vez a la columna route, obteniendo la duración del trayecto desde un punto a otro, también elimino la columna route

In [20]:
conjunto.route.explode().apply(pd.Series).head()

0   \
simulationId                         truck_id                                                      
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL   {'destination': '2e8d1b7b9fad4f083892d83c', 'd...   
                                     3953RLD   {'destination': '4bc556d929cc052029e14a78', 'd...   
                                     5030LXK   {'destination': '0550fd3ee7831440c5cf21c5', 'd...   
                                     5534TPH   {'destination': '3a471470f7f1d508af6cdfdb', 'd...   
                                     6270NFM   {'destination': '04463c1410164bb7855a830f', 'd...   

                                                                                              1   \
simulationId                         truck_id                                                      
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL   {'destination': 'f5a3aab0c10fb7c5c60f2b01', 'd...   
                                     3953RLD   {'destination': 'ce6feed06bc1a9e5aff95f04', 'd...   
                                     5030LXK   {'destination': '87c216f2b62e7b2f643ad7d4', 'd...   
                                     5534TPH   {'destination': '331afadf3cd2b8fa97eecae2', 'd...   
                                     6270NFM   {'destination': '7fbe65343adc2eb786272bfa', 'd...   

                                                                                              2   \
simulationId                         truck_id                                                      
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL   {'destination': '4b5f692a806a39d1317d15d4', 'd...   
                                     3953RLD                                                 NaN   
                                     5030LXK   {'destination': '74fe1f5b828bddbbbd194a8f', 'd...   
                                     5534TPH   {'destination': '05915a11e99fca75d7480069', 'd...   
                                     6270NFM   {'destination': 'c529324b91395971fb21755c', 'd...   

                                                                                              3   \
simulationId                         truck_id                                                      
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL   {'destination': '64bba266de127a5949f52d9a', 'd...   
                                     3953RLD                                                 NaN   
                                     5030LXK   {'destination': '66a73c577ad48cc9719b7381', 'd...   
                                     5534TPH   {'destination': '0983b6c023b45c47425f456d', 'd...   
                                     6270NFM   {'destination': '8fc6257ad9efa4296a7d3a17', 'd...   

                                                                                              4   \
simulationId                         truck_id                                                      
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL   {'destination': '6942f2624e971761b7dd9208', 'd...   
                                     3953RLD                                                 NaN   
                                     5030LXK   {'destination': '527ee974e61483070ff0ea1e', 'd...   
                                     5534TPH   {'destination': '0bf195334776828444a19f08', 'd...   
                                     6270NFM   {'destination': 'ce6feed06bc1a9e5aff95f04', 'd...   

                                                                                              5   \
simulationId                         truck_id                                                      
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL   {'destination': '103daf6394133e32f804e175', 'd...   
                                     3953RLD                                                 NaN   
                                     5030LXK   {'destination': 'ce6feed06bc1a9e5aff95f04', 'd...   
                                     5534TPH   {'destination': '3d1a2178e4ccce5da9ec05fb'

In [21]:
conjunto[["eventTime","eventType"]].explode(["eventTime","eventType"]).head()

eventTime  \
simulationId                         truck_id             
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL         90   
                                     3321FBL         90   
                                     3321FBL        660   
                                     3321FBL     751620   
                                     3321FBL     751620   

                                                              eventType  
simulationId                         truck_id                            
007f3d08-14a7-4a93-af9e-f0d9cfbcea94 3321FBL          Truck initialized  
                                     3321FBL     Truck received packets  
                                     3321FBL             Truck departed  
                                     3321FBL   Truck started delivering  
                                     3321FBL              Truck arrived

In [14]:
# planes = planes.join(planes.route.explode().apply(pd.Series), lsuffix='_sim').reset_index(drop=True).drop("route", axis=1)
# planes

Exploro la columna items_sim y me doy cuenta de que los items pueden están en diferentes localizaciones (Esta línea me ha tardado poco más de un minuto en ejecutar)

In [85]:
# planes.items_sim.explode().apply(pd.Series).locationId.apply(lambda x: x in planes.destination.to_numpy()).all()